In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [3]:
df = pd.read_excel("Dataset for Data Analytics.xlsx")

# Data Cleaning

In [4]:
print(f"Original dataset shape: {df.shape}")
print(f"Original columns: {df.columns.tolist()}")

# CLEANING PROCESS

# 1. Remove unnecessary columns
columns_to_drop = ['TrackingNumber', 'ShippingAddress', 'ItemsInCart']
df_cleaned = df.drop(columns=columns_to_drop, axis=1)
print(f"\nDropped columns: {columns_to_drop}")

# 2. Fix date format - convert to YYYY-MM-DD only
df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date']).dt.date

# 3. Standardize text columns
# Product names - capitalize first letter
df_cleaned['Product'] = df_cleaned['Product'].str.capitalize()

# Payment methods - capitalize each word
df_cleaned['PaymentMethod'] = df_cleaned['PaymentMethod'].str.title()

# Order status - capitalize first letter
df_cleaned['OrderStatus'] = df_cleaned['OrderStatus'].str.capitalize()

# Referral source - capitalize first letter
df_cleaned['ReferralSource'] = df_cleaned['ReferralSource'].str.capitalize()

# 4. Handle missing values - fill empty CouponCode with 'None'
df_cleaned['CouponCode'] = df_cleaned['CouponCode'].fillna('None')
df_cleaned['CouponCode'] = df_cleaned['CouponCode'].replace('', 'None')

# 5. Round TotalPrice to 2 decimal places
df_cleaned['TotalPrice'] = df_cleaned['TotalPrice'].round(2)

# 6. Remove any duplicate rows (based on OrderID)
df_cleaned = df_cleaned.drop_duplicates(subset=['OrderID'], keep='first')

# 7. Sort by Date for better organization
df_cleaned = df_cleaned.sort_values('Date').reset_index(drop=True)

print(f"\nCleaned dataset shape: {df_cleaned.shape}")
print(f"Cleaned columns: {df_cleaned.columns.tolist()}")

# Display first few rows
print("\nFirst 5 rows of cleaned dataset:")
print(df_cleaned.head())

# Display data types
print("\nData types after cleaning:")
print(df_cleaned.dtypes)

# Show basic statistics
print("\nBasic statistics:")
print(f"Date range: {df_cleaned['Date'].min()} to {df_cleaned['Date'].max()}")
print(f"Total unique customers: {df_cleaned['CustomerID'].nunique()}")
print(f"Total unique orders: {df_cleaned['OrderID'].nunique()}")
print(f"Products: {df_cleaned['Product'].unique().tolist()}")
print(f"Payment methods: {df_cleaned['PaymentMethod'].unique().tolist()}")
print(f"Order statuses: {df_cleaned['OrderStatus'].unique().tolist()}")
print(f"Coupon codes: {df_cleaned['CouponCode'].unique().tolist()}")
print(f"Referral sources: {df_cleaned['ReferralSource'].unique().tolist()}")

# Check for missing values
print("\nMissing values per column:")
print(df_cleaned.isnull().sum())

# Save to Excel
output_file = 'cleaned_dataset.xlsx'
df_cleaned.to_excel(output_file, index=False, sheet_name='CleanedData')
print(f"\n Cleaned dataset saved as '{output_file}'")


# Additional analysis - create summary sheet
summary_data = {
    'Metric': ['Total Orders', 'Total Revenue', 'Average Order Value', 
               'Unique Customers', 'Date Range Start', 'Date Range End',
               'Most Common Product', 'Most Common Payment Method',
               'Most Common Order Status', 'Most Common Coupon'],
    'Value': [
        len(df_cleaned),
        f"${df_cleaned['TotalPrice'].sum():,.2f}",
        f"${df_cleaned['TotalPrice'].mean():,.2f}",
        df_cleaned['CustomerID'].nunique(),
        df_cleaned['Date'].min(),
        df_cleaned['Date'].max(),
        df_cleaned['Product'].mode()[0],
        df_cleaned['PaymentMethod'].mode()[0],
        df_cleaned['OrderStatus'].mode()[0],
        df_cleaned['CouponCode'].mode()[0] if df_cleaned['CouponCode'].mode()[0] != 'None' else 'No coupon'
    ]
}

summary_df = pd.DataFrame(summary_data)

# Save summary to a separate sheet
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f"\n Summary statistics added to '{output_file}' (Sheet: 'Summary')")

# Display summary
print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)
for i, row in summary_df.iterrows():
    print(f"{row['Metric']}: {row['Value']}")

Original dataset shape: (1200, 14)
Original columns: ['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber', 'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice']

Dropped columns: ['TrackingNumber', 'ShippingAddress', 'ItemsInCart']

Cleaned dataset shape: (1200, 11)
Cleaned columns: ['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'PaymentMethod', 'OrderStatus', 'CouponCode', 'ReferralSource', 'TotalPrice']

First 5 rows of cleaned dataset:
     OrderID        Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200112  2023-01-01     C81366  Monitor         1     410.60   
1  ORD200848  2023-01-01     C54675    Chair         2     645.85   
2  ORD200236  2023-01-01     C14847  Monitor         1     318.81   
3  ORD200373  2023-01-02     C60178   Laptop         3     341.15   
4  ORD200645  2023-01-02     C82990   Laptop         2     150.05   

  PaymentMethod OrderStatus